<a href="https://colab.research.google.com/github/AileenLavelle/PBC_Object_Detection/blob/main/Tiny_Crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**This code is to just reproduce the result I got in roboflow **

 Auto-Orient: Applied Static Crop: 33-100% Horizontal Region, 28-49% Vertical Region Resize: Stretch to 2048x800

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm

!pip install sahi ultralytics
from ultralytics import YOLO
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import cv2
import matplotlib.pyplot as plt

from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.13.0.90
    Uninstalling opencv-python-4.13.0.90:
      Successfully uninstalled opencv-python-4.13.0.90
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:

#Loading YOLOv11 model
model = YOLO("yolo11n.pt")

# Setup paths
image_folder = "/content/Jupiter_Inlet"
output_folder = "/content/Jupiter_Inlet/output_yolo11n"
os.makedirs(output_folder, exist_ok=True)
ground_truth_json = "/content/Jupiter_Inlet/_annotations.coco.json"

with open(ground_truth_json) as f:
    gt = json.load(f)
#Re-generate predictions with correct category ID
predictions = []

for img_data in tqdm(gt['images'], desc="Processing images"):
    img_path = os.path.join(image_folder, img_data['file_name'])

    if not os.path.exists(img_path):
        continue

    #Change device from 'cuda:0' to 'cpu' as no GPU is available.
    results = model(img_path, conf=0.3, device='cpu', verbose=False)

    for result in results:
        boxes = result.boxes
        for i in range(len(boxes)):
            class_id = int(boxes.cls[i])
            # YOLO class 8 = boat in COCO, map to my category ID 1
            if class_id == 8:
                xyxy = boxes.xyxy[i].cpu().numpy()
                conf = float(boxes.conf[i])
                bbox = [float(xyxy[0]), float(xyxy[1]),
                        float(xyxy[2] - xyxy[0]), float(xyxy[3] - xyxy[1])]

                predictions.append({
                    "image_id": img_data['id'],
                    "category_id": 1,  # Changed from 8 to 1
                    "bbox": bbox,
                    "score": conf
                })

#Save and evaluate
pred_json = os.path.join(output_folder, "predictions.json")
with open(pred_json, 'w') as f:
    json.dump(predictions, f)

coco_gt = COCO(ground_truth_json)
coco_dt = coco_gt.loadRes(predictions) if predictions else coco_gt.loadRes([])

coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
coco_eval.params.catIds = [1]  # Evaluate category ID 1 (Boat)
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

print(f"\nTotal predictions: {len(predictions)}")
print(f"mAP@0.5:0.95: {coco_eval.stats[0]:.3f}")
print(f"mAP@0.5: {coco_eval.stats[1]:.3f}")


Processing images: 100%|██████████| 501/501 [00:51<00:00,  9.72it/s]


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.08s).
Accumulating evaluation results...
DONE (t=0.03s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.005
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.010
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.003
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.001
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.016
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.002
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.002
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets